# Lab 3b: VDB Corpus Acquisition with FAISS
**Course:** CIS 531/731 | **Module 3: Obtain**

## Objective
Take the text output generated by the Whisper ASR pipeline, convert it into semantic embeddings using a HuggingFace Transformer, and index it in a local Vector Database (FAISS).

### DevContainer Setup Instructions
Your `verify_env.py` script from MP2 already confirmed the presence of `faiss` and `transformers`. If your DevContainer failed to build those, rebuild the container now. If you are running locally outside the container, you must run:
`!pip install faiss-cpu transformers sentence-transformers`

In [3]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

print(f"FAISS Version: {faiss.__version__}")

FAISS Version: 1.16.0


### 1. Initialize the Embedding Model
We use `all-MiniLM-L6-v2` as a lightweight, highly efficient sentence embedding model to encode our transcriptions into high-dimensional vectors.

In [4]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
vector_dimension = embedder.get_sentence_embedding_dimension()
print(f"Model loaded. Embedding vector dimension: {vector_dimension}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\dacnu\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dacnu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded. Embedding vector dimension: 384


C:\Users\dacnu\AppData\Local\Temp\ipykernel_3000\3411531826.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  vector_dimension = embedder.get_sentence_embedding_dimension()


### 2. Create the FAISS Index & Ingest Corpus
In our broader project architecture, we are building a live radio stream analysis pipeline. The mock chunks below represent typical broadcast snippets you will process later in the semester.

However, for **Machine Problem 3**, we are intentionally injecting a transcribed LibriVox audiobook recording of WWI poetry into this index. We use this specific poem to guarantee a deterministic hit when we execute our semantic concept query.

In [5]:
# Initialize a flat L2 (Euclidean distance) FAISS index
index = faiss.IndexFlatL2(vector_dimension)

# Simulated incoming ASR corpus chunks (Representing our future live broadcast pipeline)
corpus = [
    "The caller sounds really upset about their recent breakup.",
    "I love this Cage the Elephant track, it reminds me of summer.",
    "Sometimes I feel like our whole generation is just out of time."
]

# === MP3 INTEGRATION TASK ===
# For this lab, we inject our deterministic LibriVox audiobook transcript.
# Paste the exact result["text"] string output from Lab 3a below:
lab3a_transcript = "PASTE_YOUR_WHISPER_TRANSCRIPT_HERE"
corpus.append(lab3a_transcript)

# Encode the text into vectors
corpus_embeddings = embedder.encode(corpus)

# Add vectors to the VDB
index.add(np.array(corpus_embeddings).astype('float32'))
print(f"Successfully ingested {index.ntotal} vectors into FAISS.")

Successfully ingested 4 vectors into FAISS.


### 3. Query the VDB (Mapping Subtle Tropes)
We query the database for a semantic concept ("doomed youth") to see which transcribed chunk matches closest in latent space.

In [6]:
query = "doomed youth"
query_embedding = embedder.encode([query]).astype('float32')

# Search the top 1 most similar result (k=1)
distances, indices = index.search(query_embedding, k=1)

matched_index = indices[0][0]
print(f"Query: '{query}'")
print(f"Closest Match in VDB: '{corpus[matched_index]}'")
print(f"L2 Distance: {distances[0][0]:.4f}")

# If your pipeline is integrated correctly, the closest match 
# should be your pasted LibriVox transcript containing Wilfred Owen's poem.

Query: 'doomed youth'
Closest Match in VDB: 'Sometimes I feel like our whole generation is just out of time.'
L2 Distance: 1.2453
